# Father — post-mortem investigation

**Question:** what activity can disk and RAM reconstruct, and what does combining them add?

Known lab context: an LD_PRELOAD compromise, with SSH access and sudo supplied.
The examination is scenario-aware; lab context is not forensic evidence.

[Investigation workflow](WORKFLOW.md)

## 0. Case and evidence

Identify the acquisitions, verify integrity, locate the root filesystem, and
establish timing and timezone context.

Hash agreement confirms consistency with the acquisition records; it does not
prove acquisition completeness.

In [1]:
import json
from pathlib import Path

RUN_ID = "father-u22-20260820-01"
REPO_ROOT = Path("../..")
RUN_DIR = REPO_ROOT / "shared/experiments" / RUN_ID
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
acquisition = json.loads((RUN_DIR / "dumps/acquisition.json").read_text())
assert manifest["run_id"] == RUN_ID, "Manifest run mismatch"
DISK_IMAGE = RUN_DIR / "dumps" / acquisition["disk"]["path"]
MEMORY_IMAGE = RUN_DIR / "dumps" / acquisition["memory"]["path"]
for evidence in (DISK_IMAGE, MEMORY_IMAGE):
    assert evidence.is_file(), f"Missing evidence: {evidence}"

In [2]:
print("Run:", RUN_ID)
print("Recorded platform:", manifest["platform"])
print("Acquisition repository:", manifest["repository"])
for kind, evidence in (("disk", DISK_IMAGE), ("memory", MEMORY_IMAGE)):
    print(kind, evidence)
    print("Recorded size:", acquisition[kind]["size_bytes"], "SHA-256:", acquisition[kind]["sha256"])
print("Metadata only; fresh integrity checks follow.")

Run: father-u22-20260820-01
Recorded platform: {'distro_id': 'ubuntu-22.04', 'guest_os': 'Ubuntu 22.04.5 LTS', 'kernel': '5.15.0-179-generic', 'profile': 'vanilla', 'timezone': 'Etc/UTC'}
Acquisition repository: {'commit': 'f5425b29d3e2b2d622bb6c2e78bf383c199ee4f8', 'working_tree': 'clean'}
disk ../../shared/experiments/father-u22-20260820-01/dumps/disk/evidence_disk.E01
Recorded size: 10737418240 SHA-256: a55b62906122609698bd1b9609aa1deffee8ec84d68be9d28318e99123508226
memory ../../shared/experiments/father-u22-20260820-01/dumps/memory/mem.raw
Recorded size: 2147747795 SHA-256: f3e90371fbedf1e7573fabfabe788ce529fae92cb7a93a6d6645dc600a52643b
Metadata only; fresh integrity checks follow.


### Output preservation and examiner environment

- `data/`: complete analysis-tool output and nonempty error output.
- `recovered/`: extracted inode or data-block content, requiring validation.

Numbered files follow the examination order below; `version-*` files record tool
versions. Examination tools use UTC; the examiner's timezone is recorded separately.


In [3]:
from datetime import datetime, timezone
from functools import partial
from investigation_utils import get_rootfs_offset, run_command as execute_command, parse_istat

EXAM_DIR = REPO_ROOT / "shared/investigations" / RUN_ID
DATA_DIR = EXAM_DIR / "data"
RECOVERED_DIR = EXAM_DIR / "recovered"
for directory in (DATA_DIR, RECOVERED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Default destination; pass out_dir=RECOVERED_DIR to override it for one call.
run_command = partial(execute_command, out_dir=DATA_DIR)
print("Analysis output:", DATA_DIR)
print("Extracted content:", RECOVERED_DIR)

Analysis output: ../../shared/investigations/father-u22-20260820-01/data
Extracted content: ../../shared/investigations/father-u22-20260820-01/recovered


In [4]:
print("Examiner timezone:", datetime.now().astimezone().tzinfo)
print("Examiner /etc/localtime:", Path("/etc/localtime").resolve())

Examiner timezone: CEST
Examiner /etc/localtime: /usr/share/zoneinfo/Europe/Rome


In [5]:
for tool, flag in (("ewfverify", "-V"), ("mmls", "-V"),
                   ("vol3", "-h"), ("log2timeline", "--version")):
    version = run_command([tool, flag], label=f"version-{tool}.txt")
    print((version.stdout + version.stderr).splitlines()[0])

$ ewfverify -V
ewfverify 20240506
$ mmls -V
The Sleuth Kit ver 4.15.0
$ vol3 -h
Volatility 3 Framework 2.28.2
$ log2timeline --version
plaso - log2timeline version 20260512


### Acquisition timing — recorded provenance

**Use UTC for comparison; a timezone label does not establish clock accuracy.**
The acquisition code was checked at the run's recorded revision `f5425b29d3e2b2d622bb6c2e78bf383c199ee4f8`.
These records describe acquisition provenance, not independently observed attack activity.

| Record | Time convention and meaning |
|---|---|
| Manifest `timestamps.*` | Explicit UTC (`Z`), recorded on the host. Scenario start/end delimit the orchestration phase; run end follows acquisition. |
| `starting_snapshot.created_at` | Libvirt snapshot creation time, converted to UTC; not this run's acquisition time. |
| `platform.timezone` | Guest timezone (`Etc/UTC`), not the acquisition host timezone. |
| Manifest `date` | Host-local calendar date used for identification; not a timestamp for chronological comparison. |
| Disk `timestamp` | Unix epoch seconds, recorded after EWF verification and hash-file writing; not the disk capture instant. |

The EWF console strings have no zone suffix. The image header below is displayed
in UTC and provides the EWF packaging time. Preserve the original console strings;
do not relabel them as UTC or apply a blanket two-hour correction.

**Limit:** successful RAM acquisition records omit start/end times. The recorded
pipeline is RAM → pre-shutdown callback → guest shutdown → offline disk acquisition.
The exact capture gap and host/guest clock error remain unmeasured.


In [6]:
print("Times in UTC, displayed to milliseconds:")
for label, value in [
    ("Baseline snapshot created", manifest["starting_snapshot"]["created_at"]),
    ("Scenario started", manifest["timestamps"]["scenario_started_at"]),
    ("Scenario ended", manifest["timestamps"]["scenario_ended_at"]),
    ("Disk metadata finalized", datetime.fromtimestamp(acquisition["disk"]["timestamp"], timezone.utc).isoformat()),
    ("Run ended", manifest["timestamps"]["run_ended_at"]),
]:
    event_time = datetime.fromisoformat(value).astimezone(timezone.utc)
    print(f"{label:25} {event_time.isoformat(sep=' ', timespec='milliseconds')}")

print("\nRecorded guest timezone:", manifest["platform"]["timezone"])
print("RAM capture start/end and exact RAM-to-disk gap: not recorded.")

Times in UTC, displayed to milliseconds:
Baseline snapshot created 2026-08-11 17:14:03.000+00:00
Scenario started          2026-08-20 15:11:24.514+00:00
Scenario ended            2026-08-20 15:12:56.183+00:00
Disk metadata finalized   2026-08-20 15:13:47.585+00:00
Run ended                 2026-08-20 15:13:47.586+00:00

Recorded guest timezone: Etc/UTC
RAM capture start/end and exact RAM-to-disk gap: not recorded.


In [7]:
ewf_timing = run_command(["ewfinfo", DISK_IMAGE], label="01-ewfinfo-timing.txt")
print("EWF header dates displayed in UTC:")
for line in ewf_timing.stdout.splitlines():
    if "date:" in line:
        print(line.strip())

$ ewfinfo ../../shared/experiments/father-u22-20260820-01/dumps/disk/evidence_disk.E01
EWF header dates displayed in UTC:
Acquisition date:	Thu Aug 20 15:13:07 2026
System date:		Thu Aug 20 15:13:07 2026


**Observation:** EWF acquisition/system dates display `2026-08-20 15:13:07`
in UTC. The original acquisition console reports `17:13:07` without a timezone
suffix. The two-hour difference is consistent with UTC+02:00 formatting; it does
not establish the host's timezone name or clock accuracy.

Disk metadata was finalized at `15:13:47.585654 UTC`, after EWF verification and
hash-file writing. This timestamp does not identify when the disk state was captured.


### Disk and RAM integrity

**Question:** do the acquired images still match their acquisition records?

- **Disk:** check EWF segment sizes, then use `ewfverify` to verify the disk
  media-stream SHA-256 across all segments. This is not an E01 container-file hash.
- **RAM:** check the file size and SHA-256, then identify the dump format.

The RAM record's historical `verified: false` flag does not describe these fresh checks.

In [8]:
for segment in acquisition["disk"]["segment_metadata"]:
    segment_path = RUN_DIR / "dumps" / segment["path"]
    segment_size = segment_path.stat().st_size
    assert segment_size == segment["size_bytes"], f"Size mismatch: {segment_path}"
    print(f"{segment_path.name}: {segment_size:,} bytes — size matches")

evidence_disk.E01: 1,572,848,856 bytes — size matches
evidence_disk.E02: 333,454,177 bytes — size matches


In [9]:
verify_proc = run_command(
    ["ewfverify", "-d", "sha256", DISK_IMAGE], label="02-ewfverify.txt"
)
sha_lines = [line for line in verify_proc.stdout.splitlines()
             if line.startswith("SHA256 hash calculated over data:")]
assert len(sha_lines) == 1, "Inspect the full verification output before continuing"
computed_disk_sha256 = sha_lines[0].split(":", 1)[1].strip()
assert computed_disk_sha256 == acquisition["disk"]["sha256"], "Disk hash mismatch"
print("Disk SHA-256 matches acquisition record:", computed_disk_sha256)

$ ewfverify -d sha256 ../../shared/experiments/father-u22-20260820-01/dumps/disk/evidence_disk.E01
Disk SHA-256 matches acquisition record: a55b62906122609698bd1b9609aa1deffee8ec84d68be9d28318e99123508226


In [10]:
assert MEMORY_IMAGE.stat().st_size == acquisition["memory"]["size_bytes"], "RAM size mismatch"
memory_hash = run_command(["sha256sum", MEMORY_IMAGE], label="03-memory-sha256.txt")
computed_memory_sha256 = memory_hash.stdout.split()[0]
assert computed_memory_sha256 == acquisition["memory"]["sha256"], "RAM hash mismatch"
print("RAM size and SHA-256 match acquisition record:", computed_memory_sha256)

$ sha256sum ../../shared/experiments/father-u22-20260820-01/dumps/memory/mem.raw
RAM size and SHA-256 match acquisition record: f3e90371fbedf1e7573fabfabe788ce529fae92cb7a93a6d6645dc600a52643b


In [11]:
memory_format = run_command(["file", MEMORY_IMAGE], label="04-memory-format.txt")
print(memory_format.stdout)

$ file ../../shared/experiments/father-u22-20260820-01/dumps/memory/mem.raw
../../shared/experiments/father-u22-20260820-01/dumps/memory/mem.raw: ELF 64-bit LSB core file, x86-64, version 1 (SYSV), SVR4-style



**Limit:** hash agreement confirms consistency with the acquisition records,
not capture completeness. Dump format identification does not establish
Volatility compatibility; symbol validation belongs to the memory examination.

### Partition and root-filesystem orientation

Select the partition offset manually from `mmls`; `fsstat` identifies the
filesystem at that offset. Here, GPT partition 000 starts at sector **227328**,
with **512-byte sectors**.

Use the filesystem label and `/etc/fstab` to confirm its root role.
Read `/etc/timezone` for the guest timezone setting. All commands read the image
directly, without mounting it.


In [12]:
mmls_proc = run_command(["mmls", DISK_IMAGE], label="05-mmls.txt")
print(mmls_proc.stdout)
offset_sector = get_rootfs_offset(DISK_IMAGE)
if offset_sector is None:
    raise ValueError("No Ext4 partition found in the disk image.")
sector_size = 512
print("Selected offset:", offset_sector, "sectors =", int(offset_sector) * sector_size, "bytes")

$ mmls ../../shared/experiments/father-u22-20260820-01/dumps/disk/evidence_disk.E01
GUID Partition Table (EFI)
Offset Sector: 0
Units are in 512-byte sectors

      Slot      Start        End          Length       Description
000:  Meta      0000000000   0000000000   0000000001   Safety Table
001:  -------   0000000000   0000002047   0000002048   Unallocated
002:  Meta      0000000001   0000000001   0000000001   GPT Header
003:  Meta      0000000002   0000000033   0000000032   Partition Table
004:  013       0000002048   0000010239   0000008192   
005:  014       0000010240   0000227327   0000217088   
006:  000       0000227328   0020971486   0020744159   
007:  -------   0020971487   0020971519   0000000033   Unallocated

Selected offset: 0000227328 sectors = 116391936 bytes


In [ ]:
fsstat_proc = run_command(["fsstat", "-o", offset_sector, DISK_IMAGE], label="06-fsstat.txt")
print("\n".join(fsstat_proc.stdout.splitlines()[:40]))
print("Full filesystem output:", DATA_DIR / "06-fsstat.txt")

In [ ]:
for path, label in (("/etc/fstab", "07-fstab.txt"),
                    ("/etc/timezone", "08-timezone.txt")):
    extracted = run_command(["fcat", "-o", offset_sector, path, DISK_IMAGE], label=label)
    print(path, "\n", extracted.stdout[:4000])

### Operating system release

Read `/usr/lib/os-release` from the disk image to identify the installed
operating system before examining the compromise.

In [ ]:
os_release = run_command(
    ["fcat", "-o", offset_sector, "/usr/lib/os-release", DISK_IMAGE],
    label="09-os-release.txt",
)
print(os_release.stdout)

### Section 0 interpretation

| Orientation check | Observation and supporting output |
|---|---|
| Acquisition timing | EWF header dates display `2026-08-20 15:13:07 UTC` (`data/01-ewfinfo-timing.txt`). Disk metadata was finalized later, at `15:13:47.585654 UTC`. |
| Disk integrity | EWF media-stream SHA-256 matches the acquisition record (`data/02-ewfverify.txt`); both segment sizes match in the checks above. |
| RAM integrity and format | SHA-256 and size (2,147,747,795 bytes) match the acquisition record; `file` identifies an x86-64 ELF core (`data/03-memory-sha256.txt`, `data/04-memory-format.txt`). |
| Root filesystem | GPT partition 000 starts at sector 227328, with 512-byte sectors (`data/05-mmls.txt`). Ext4 uses 4096-byte blocks and label `cloudimg-rootfs` (`data/06-fsstat.txt`); `/etc/fstab` assigns that label to `/` (`data/07-fstab.txt`). |
| Guest timezone | `/etc/timezone` contains `Etc/UTC`, matching the manifest (`data/08-timezone.txt`). |
| Operating system | `/usr/lib/os-release` identifies Ubuntu 22.04.5 LTS, consistent with the manifest (`data/09-os-release.txt`). |

**Integrity limit:** hash agreement establishes consistency with the recorded
images, not acquisition completeness or readiness for RAM analysis.

**Timing limit:** the disk metadata timestamp follows processing; it is not the
disk capture instant. RAM capture start/end times are absent, so the exact
RAM-to-disk gap cannot be calculated. UTC makes timestamps comparable but does
not establish host/guest clock accuracy.


## 1. Follow the preload configuration

**Question:** which library is configured, and what can disk establish about it?

Retained disk commands only; review after section 0. Read the configuration,
follow its library reference, inspect metadata, and characterise the extracted
object. No identity is taken from the scenario inputs during this examination.

In [ ]:
# Resolve the path to an inode on this run's own image.'
ifind_proc = run_command(
    ["ifind", "-o", offset_sector, "-n", "/etc/ld.so.preload", str(DISK_IMAGE)],
    label="04-ifind-preload.txt"
)
preload_inode = ifind_proc.stdout.strip()
assert preload_inode, "/etc/ld.so.preload did not resolve to an inode on this image"
print("/etc/ld.so.preload inode:", preload_inode)

# icat reads its content (the installed library's path).
icat_proc = run_command(
    ["icat", "-o", offset_sector, str(DISK_IMAGE), preload_inode],
    label="05-ld.so.preload-content.txt"
)
preload_content = icat_proc.stdout
print("content:", repr(preload_content))

# istat reads its MAC times.
istat_proc = run_command(
    ["istat", "-o", offset_sector, str(DISK_IMAGE), preload_inode],
    label="06-istat-preload.txt"
)
preload_istat = parse_istat(istat_proc.stdout)
print("istat:", preload_istat)

### Follow the library reference

This inherited cell assumes one absolute `/lib/…` entry. Review that assumption
against the actual preload content before adapting it to another run. Path and
symlink handling remain draft code, not a general Linux resolver.

In [ ]:
# Discover /lib symlink target from this run's own evidence.
lib_dir_ifind = run_command(
    ["ifind", "-o", offset_sector, "-n", "/lib", str(DISK_IMAGE)],
    label="07a-ifind-lib-dir.txt"
)
lib_dir_inode = lib_dir_ifind.stdout.strip()
assert lib_dir_inode, "/lib did not resolve to an inode on this image"

lib_dir_istat = run_command(
    ["istat", "-o", offset_sector, str(DISK_IMAGE), lib_dir_inode],
    label="07b-istat-lib-dir.txt"
)
lib_dir_info = parse_istat(lib_dir_istat.stdout)
print("/lib symlink target:", lib_dir_info["symlink_target"])

# Resolve library path (handling the /lib symlink discovered above).
lib_path = preload_content.strip()
if lib_dir_info["symlink_target"]:
    # Handle Usr-Merge (e.g. /lib -> /usr/lib) by resolving symlink target.
    lib_name = lib_path.split("/")[-1]
    lib_path_resolved = "/" + lib_dir_info["symlink_target"] + "/" + lib_name
else:
    lib_path_resolved = lib_path
print(f"Resolved library path: {lib_path_resolved}")

# Path Resolution -- locate the installed library inode by resolved path.
lib_ifind = run_command(
    ["ifind", "-o", offset_sector, "-n", lib_path_resolved, str(DISK_IMAGE)],
    label="08-ifind-lib.txt"
)
lib_inode = lib_ifind.stdout.strip()
print(f"Library inode: {lib_inode}")
assert lib_inode, f"{lib_path_resolved} did not resolve to an inode"

# Extract the installed library's bytes for hashing.
lib_icat = run_command(
    ["icat", "-o", offset_sector, str(DISK_IMAGE), lib_inode],
    label="09-installed-lib.bin", out_dir=RECOVERED_DIR
)

import hashlib
lib_sha256 = hashlib.sha256((RECOVERED_DIR / "09-installed-lib.bin").read_bytes()).hexdigest()
print("Extracted library SHA-256:", lib_sha256)

### Characterise the extracted object

`file` and `strings` provide leads, not proof of executed hooks. Review the
complete saved strings output as needed; the display below is just an excerpt.
Timestamps are reported for interpretation, without an automatic timestomp flag.

In [ ]:
installed_lib_bin = RECOVERED_DIR / "09-installed-lib.bin"
file_proc = run_command(["file", "-b", str(installed_lib_bin)], label="25-file-installed-lib.txt")
print(file_proc.stdout)
strings_proc = run_command(
    ["strings", "-a", "-n", "8", str(installed_lib_bin)], label="14-strings-installed-lib.txt"
)
print("\n".join(strings_proc.stdout.splitlines()[:30]))
print("Full strings output:", DATA_DIR / "14-strings-installed-lib.txt")

In [ ]:
lib_istat_proc = run_command(
    ["istat", "-o", offset_sector, str(DISK_IMAGE), lib_inode], label="10-istat-lib.txt"
)
print(lib_istat_proc.stdout)

**Interpretation — pending.** Record observations, alternative explanations, and
selected path/time pivots. A library on disk does not establish runtime loading.

## 2. Establish the surrounding activity

**Question:** what local artifacts and records explain the surrounding activity?

Retained starting point: enumerate `/tmp` without selecting scenario marker names.
Content examination, relevant accounts/history, auth/service logs, wtmp/btmp,
and the initial Plaso examination are still to be built one section at a time.
Plaso extraction must preserve parser/extraction information and original record
locators. Native log checks may verify the same underlying records, not add
independent evidence. Investigate within a justified, documented time window.

In [ ]:
# Path Resolution
tmp_inode = run_command(
    ["ifind", "-o", offset_sector, "-n", "/tmp", str(DISK_IMAGE)],
    label="10-ifind-tmp.txt",
).stdout.strip()
assert tmp_inode, "/tmp did not resolve to an inode on this image"
print("/tmp inode:", tmp_inode)

# File Listing -- fls started directly from /tmp's own already-resolved inode
# (recursive); the standard TSK approach for a known directory.
tmp_listing_result = run_command(
    ["fls", "-o", offset_sector, "-r", "-p", str(DISK_IMAGE), tmp_inode],
    label="11-fls-tmp.txt",
)
print(tmp_listing_result.stdout)

**Interpretation — pending.** Select candidates from the displayed evidence,
then inspect only justified content/metadata. Do not infer malice from a name
or live hiding from offline visibility. Add the initial sourced chronology here.

## 3. Follow the compromise into memory

**Not implemented.** Use the observed library/path and service context to examine
process mappings, ancestry, credentials, command lines, and sockets. Resolve the
matching symbols first. History or retained-content examination depends on the
actual process type and available structures. Avoid scenario-supplied PIDs/ports.

Output: supported runtime observations and new pivots, not automatic malware labels.

## 4. Revisit disk and missing artifacts

**Question:** what can metadata, ext4 journal reconstruction and content recovery
establish about missing/deleted material?

Deletion recovery is fully in scope. Begin with deleted-entry enumeration, then
follow the bounded journal/carving test plan in [RECOVERY_NOTES.md](RECOVERY_NOTES.md).
An empty listing does not end the investigation. Keep content, metadata-only,
bounded negative and tool-failure outcomes distinct, with preserved raw locators.
Validate candidates and document any known-input assistance before assigning
claim support or counting validated artifacts. Integrate ordinary tool commands
with the existing helper; do not build a custom recovery engine.


In [ ]:
# Deleted File Listing -- recursive fls restricted to DELETED entries only
# (-r recurse, -d deleted-only), scoped to the /tmp inode from Section 8.
fls_deleted_tmp = run_command(
    ["fls", "-o", offset_sector, "-r", "-d", "-p", str(DISK_IMAGE), tmp_inode],
    label="13-fls-deleted-tmp.txt",
)
print("deleted directory entries under /tmp (fls -r -d):")
print(fls_deleted_tmp.stdout or "(empty -- no deleted entry listed)")

**Interpretation — pending.** A deleted name, metadata, and recovered content are
different observations. Empty output is not proof that recovery is impossible.

## 5. Assemble the incident chronology

**Not implemented. Plaso is required.** Revisit its events using the observed
paths, accounts, and times. Combine selected original disk/log events with
relevant RAM observations in one sourced chronology. Preserve timestamp meaning,
precision, unknowns, and conflicts. Do not inject scenario times to fill gaps.

Columns: time/interval | observation | original evidence locator | interpretation/limit.

## 6. Locked result tables

**Implementation pending; methodology approved on 2026-09-10.**
[RESULTS_PREVIEW.md](RESULTS_PREVIEW.md) is the immutable specification. Its
values remain fictional and must never be copied into this run's results.

Produce: (1) Chronology; (2) Claim Coverage & Source Support;
(3) Multi-Source Contribution; (4) Footprint Inventory.

Use a documented GT claim list with exact predicates and record references.
Manually assign S/P/U/N/A and combined conclusions from observed evidence;
retain locators, underlying origins and missing elements. Calculate the locked
counts with simple Python, preserving source independence and deduplication.
Recovery outcomes belong in these same tables. Metric selection is closed.

Compare later runs only with fixed treatment and claim definitions; re-examine
all observations rather than copying verdicts or interpreting one run as a
general distribution effect.


## 7. Validate against the controlled scenario

**Not implemented.** Once the reconstruction is written, compare it with the
frozen run's command log and input identities. Disclose prior knowledge and
assisted checks. Identify agreements, discrepancies, and unresolved actions.
Ground truth is an experimental reference, not another forensic evidence source.